<a href="https://colab.research.google.com/github/Kshitij8097/UofT_Machine_Learning_3253/blob/main/Auto_theft_(Supervised_with_label)_Kshitij_copy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Business Problem**

As a auto-theft insurer that provides (6 months / 1 year) insurance, I want to be able to determine if the location of my customer is high/low risk so that I will be able to determine the appropriate premium.

*I added a time period since a location may not be perpetually low or high risk since the situation of a location may change overtime.


In [ ]:
import numpy as np
import pandas as pd
import requests
import re

# DATA PREPARATION

## Data Cleaning — Step 1: Load data + clean coordinates




In [8]:
"""
Renaming columns -
We rename columns like LAT_WGS84 → LAT to make the code easier to read later.

Removing invalid coordinates -
dropna(subset=["LAT", "LONG"]) removes rows where latitude or longitude is missing.
(df["LAT"] != 0) removes rows where latitude is 0.
(df["LONG"] != 0) removes rows where longitude is 0.

Coordinates (0,0) are in the Atlantic Ocean near Africa — not Toronto — so they must be removed.
"""

import pandas as pd
import numpy as np

# 1. Load the dataset
df = pd.read_csv("Auto_Theft_Open_Data.csv")

# 2. Rename columns for consistency (optional but helpful)
df = df.rename(columns={
    "LAT_WGS84": "LAT",
    "LONG_WGS84": "LONG",
    "NEIGHBOURHOOD_158": "NEIGHBOURHOOD",
    "HOOD_158": "HOOD"
})

# 3. Remove rows with missing or invalid coordinates
df = df.dropna(subset=["LAT", "LONG"])   # removes rows where LAT or LONG is NaN
df = df[(df["LAT"] != 0) & (df["LONG"] != 0)]  # removes rows where LAT or LONG = 0

# 4. Reset index after cleaning
df = df.reset_index(drop=True)

# Show the cleaned shape
df.shape

(75906, 32)

In [9]:
len(df)

75906

## Step 2 — Convert OCC_DATE into a clean datetime

In [10]:
"""
Dataset has inconsistent date formats:
01/01/2014 05:00:00
12/25/2013 5:00:00 AM
1/13/2014 5:00:00 AM
"""

# 1. Convert OCC_DATE to datetime
df["OCC_DATE"] = pd.to_datetime(df["OCC_DATE"], errors="coerce")

# 2. Remove rows where OCC_DATE could not be parsed
df = df.dropna(subset=["OCC_DATE"])

# 3. Extract clean time features
df["OCC_YEAR"] = df["OCC_DATE"].dt.year
df["OCC_MONTH"] = df["OCC_DATE"].dt.month
df["OCC_DAY"] = df["OCC_DATE"].dt.day
df["OCC_DOW"] = df["OCC_DATE"].dt.dayofweek   # Monday=0, Sunday=6
df["OCC_HOUR"] = df["OCC_DATE"].dt.hour

# 4. Reset index again
df = df.reset_index(drop=True)

df.head()


,OBJECTID,EVENT_UNIQUE_ID,REPORT_DATE,OCC_DATE,REPORT_YEAR,REPORT_MONTH,REPORT_DAY,REPORT_DOY,REPORT_DOW,REPORT_HOUR,...,CSI_CATEGORY,HOOD,NEIGHBOURHOOD,Region,HOOD_140,NEIGHBOURHOOD_140,LONG,LAT,x,y
0,1,GO-20141262837,01/01/14 5:00,2013-12-25 05:00:00,2014,January,1,1,Wednesday,15,...,Auto Theft,159,Etobicoke City Centre (159),Region. 1,14,Islington-City Centre West (14),-79.529692,43.618988,-8853204.784,5406667.722
1,2,GO-20141263217,01/01/14 5:00,2013-12-31 05:00:00,2014,January,1,1,Wednesday,16,...,Auto Theft,43,Victoria Village (43),NaN,43,Victoria Village (43),-79.306754,43.734654,-8828387.423,5424470.688
2,10,GO-20141275619,01/03/14 5:00,2013-12-20 05:00:00,2014,January,3,3,Friday,17,...,Auto Theft,157,Bendale South (157),NaN,127,Bendale (127),-79.248540,43.748432,-8821907.125,5426593.568
3,19,GO-20141282861,01/04/14 5:00,2013-12-25 05:00:00,2014,January,4,4,Saturday,21,...,Auto Theft,163,Fort York-Liberty Village (163),NaN,82,Niagara (82),-79.401144,43.636915,-8838894.956,5409424.778
4,30,GO-20141293197,01/06/14 5:00,2013-12-28 05:00:00,2014,January,6,6,Monday,20,...,Auto Theft,152,East Willowdale (152),NaN,51,Willowdale East (51),-79.397562,43.779308,-8838496.192,5431352.787


In [11]:
len(df)

45818

# Step 3  - creating grid cells for location‑based risk

In [12]:
# 1. Define grid size
GRID_SIZE = 0.005   # medium grid

# 2. Create grid cell coordinates by flooring lat/long
df["GRID_LAT"] = (df["LAT"] // GRID_SIZE).astype(int)
df["GRID_LONG"] = (df["LONG"] // GRID_SIZE).astype(int)

# 3. Create a combined grid ID
df["GRID_ID"] = df["GRID_LAT"].astype(str) + "_" + df["GRID_LONG"].astype(str)

# 4. Count thefts per grid cell
grid_counts = df.groupby("GRID_ID").size().reset_index(name="GRID_THEFT_COUNT")

# 5. Merge theft counts back into main dataframe
df = df.merge(grid_counts, on="GRID_ID", how="left")

# 6. Compute percentile rank for each grid cell
df["GRID_PERCENTILE"] = df["GRID_THEFT_COUNT"].rank(pct=True)

# 7. Create IS_HIGH_RISK_GRID (top 20% = high risk)
df["IS_HIGH_RISK_GRID"] = (df["GRID_PERCENTILE"] >= 0.80).astype(int)

df.head()

,OBJECTID,EVENT_UNIQUE_ID,REPORT_DATE,OCC_DATE,REPORT_YEAR,REPORT_MONTH,REPORT_DAY,REPORT_DOY,REPORT_DOW,REPORT_HOUR,...,LONG,LAT,x,y,GRID_LAT,GRID_LONG,GRID_ID,GRID_THEFT_COUNT,GRID_PERCENTILE,IS_HIGH_RISK_GRID
0,1,GO-20141262837,01/01/14 5:00,2013-12-25 05:00:00,2014,January,1,1,Wednesday,15,...,-79.529692,43.618988,-8853204.784,5406667.722,8723,-15906,8723_-15906,11,0.124351,0
1,2,GO-20141263217,01/01/14 5:00,2013-12-31 05:00:00,2014,January,1,1,Wednesday,16,...,-79.306754,43.734654,-8828387.423,5424470.688,8746,-15862,8746_-15862,23,0.446135,0
2,10,GO-20141275619,01/03/14 5:00,2013-12-20 05:00:00,2014,January,3,3,Friday,17,...,-79.248540,43.748432,-8821907.125,5426593.568,8749,-15850,8749_-15850,9,0.082337,0
3,19,GO-20141282861,01/04/14 5:00,2013-12-25 05:00:00,2014,January,4,4,Saturday,21,...,-79.401144,43.636915,-8838894.956,5409424.778,8727,-15881,8727_-15881,24,0.469172,0
4,30,GO-20141293197,01/06/14 5:00,2013-12-28 05:00:00,2014,January,6,6,Monday,20,...,-79.397562,43.779308,-8838496.192,5431352.787,8755,-15880,8755_-15880,19,0.336112,0
